# XGBoost Notebook - Line by Line Explanation

This walks through every line of code in `ml_xgboost_optimized.ipynb` and explains what it does and why.

---
## Cell 1: Imports

```python
import pandas as pd
```
Loads the **pandas** library and gives it the shortcut name `pd`. Pandas is what reads CSV files and stores data in tables (called DataFrames). Every time you see `pd.something`, it's using pandas.

```python
import numpy as np
```
Loads **NumPy**, a math library. Shortcut `np`. Used here mostly for `np.nan` (a special "blank" value) and `np.random.seed`.

```python
import matplotlib.pyplot as plt
```
Loads **matplotlib**, the standard Python charting library. `plt` is the shortcut. Used later to display the confusion matrix.

```python
from xgboost import XGBClassifier
```
Pulls out the **XGBClassifier** class from the xgboost library. This is the actual machine learning model. A "classifier" means it sorts things into categories (Normal vs Impaired), as opposed to predicting a number.

```python
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
```
From scikit-learn, pulls out three tools:
- `train_test_split` - splits data into training set and test set
- `RandomizedSearchCV` - tries random combinations of model settings and picks the best one
- `StratifiedKFold` - a way to split data for cross-validation that keeps the same ratio of Normal/Impaired in each split

```python
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
```
Tools to measure how good the model is:
- `accuracy_score` - what % of predictions were correct
- `classification_report` - detailed breakdown per class (precision, recall, f1)
- `confusion_matrix` - the 2x2 grid of correct vs wrong predictions
- `ConfusionMatrixDisplay` - draws the confusion matrix as a visual chart

```python
import shap
```
Loads the **SHAP** library. This is what explains the model's decisions after training, producing those colored dot charts.

```python
%matplotlib inline
```
A Jupyter-specific command. It tells the notebook to display charts directly inside the notebook instead of opening a separate window.

```python
np.random.seed(42)
```
Locks the random number generator to a fixed starting point. This means every time you run the notebook, the random splits, random search, and random samples will be exactly the same. Without this, results would change slightly every run. The number 42 is arbitrary (it's a meme from Hitchhiker's Guide).

---
## Cell 2: Column mapping dictionary

```python
column_mapping = {
    'NACCID': 'Patient_ID',
    'VISITYR': 'Visit_Year',
    ...
}
```
This is just a Python dictionary (a lookup table). The left side is the original column name in the CSV, the right side is what it should be called instead. Nothing actually happens to the data yet. This dictionary gets used in the next cell.

---
## Cell 3: Loading the data

```python
file_path = '../data/investigator_nacc72.csv'
```
Stores the file location in a variable. `../` means "go up one folder" (from `/exploration/` up to the project root, then into `/data/`).

```python
df = pd.read_csv(file_path, nrows=150000, low_memory=False)
```
Reads the CSV file. `nrows=150000` means only read the first 150,000 rows instead of the whole file (which is millions of rows and would be slow). `low_memory=False` tells pandas to figure out the data types in one pass instead of guessing, which avoids warnings. The result is stored in `df` (short for DataFrame, which is basically a big spreadsheet in memory).

```python
df = df.rename(columns=column_mapping)
```
Takes the DataFrame and renames every column that appears in the `column_mapping` dictionary. So the column called `NACCAGE` becomes `Age_At_Visit`, `DEP` becomes `Depression`, etc. Columns not in the dictionary keep their original names.

```python
print(f"Loaded and renamed: {df.shape}")
```
`df.shape` returns something like `(150000, 1024)` meaning 150,000 rows and 1,024 columns. The `f"..."` is an f-string, which lets you put variables inside curly braces `{}`.

```python
except FileNotFoundError:
    df = pd.DataFrame()
```
If the CSV file doesn't exist at that path, instead of crashing, it creates an empty DataFrame so the rest of the notebook can gracefully skip instead of throwing errors everywhere.

---
## Cell 4: Preparing the data

```python
if not df.empty:
```
Only runs the code below if data was successfully loaded. `df.empty` is True when the DataFrame has zero rows.

```python
df_baseline = df.sort_values(by=['Patient_ID', 'Visit_Year']).drop_duplicates(subset=['Patient_ID'], keep='first')
```
This is two operations chained together:
1. `.sort_values(by=['Patient_ID', 'Visit_Year'])` - sorts all 150k rows first by patient, then by year within each patient
2. `.drop_duplicates(subset=['Patient_ID'], keep='first')` - for each patient, keeps only the first row (their earliest visit) and drops the rest

Result: one row per patient, representing their baseline health when they first entered the study.

```python
target = 'Cognitive_Status'
```
The column the model is trying to predict. Values are 1 (Normal), 2 (Impaired, not MCI), 3 (MCI), 4 (Dementia).

```python
features = [
    'Age_At_Visit', 'Sex_1M_2F', 'Years_Education',
    'APOE4_Allele_Count',
    ...
]
```
A list of all the column names the model will use as inputs. These are the 37 health/lifestyle variables that are being fed in as clues.

```python
existing_features = [f for f in features if f in df_baseline.columns]
```
A safety check. This goes through the features list and only keeps the ones that actually exist in the data. If a column got renamed differently or doesn't exist, it gets silently skipped instead of crashing.

```python
model_data = df_baseline[existing_features + [target]].copy()
```
Creates a new, smaller DataFrame with only the columns needed (the 37 features + the target). `.copy()` makes an independent copy so changes don't accidentally affect the original `df_baseline`.

```python
model_data = model_data[model_data[target].isin([1, 2, 3, 4])]
```
Keeps only rows where Cognitive_Status is 1, 2, 3, or 4. Some rows might have other codes (like 0 or -4 for missing), and those get dropped.

```python
model_data[existing_features] = model_data[existing_features].mask(model_data[existing_features] < 0, np.nan)
```
In the NACC dataset, negative numbers (like -4) mean "not recorded" or "not applicable." `.mask()` finds every cell with a value below 0 and replaces it with `NaN` (Not a Number, basically "blank").

```python
model_data = model_data.fillna(model_data.median())
```
Fills in all those blanks with the median (middle value) of that column. For example, if the median BMI is 27, every patient with missing BMI gets 27. This is a simple imputation strategy.

---
## Cell 5: Training the model (the big one)

```python
y = (model_data[target] > 1).astype(int)
```
Creates the labels the model will try to predict. `model_data[target] > 1` creates True/False for each row (True if the patient has any impairment). `.astype(int)` converts True to 1 and False to 0. So now: 0 = Normal, 1 = Impaired.

```python
X = model_data[existing_features]
```
`X` is the standard variable name for "inputs" in machine learning. This grabs just the 37 feature columns (no target).

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
```
Splits the data into two groups:
- **Training set** (80%): The model learns from this.
- **Test set** (20%): Held back, used only at the end to check accuracy.

`stratify=y` makes sure both groups have the same ratio of Normal to Impaired patients. Without this, the test set might accidentally end up with mostly Normal patients.

`random_state=42` makes the split the same every time you run it.

```python
scale_ratio = (y_train == 0).sum() / (y_train == 1).sum()
```
Counts how many Normal patients there are for every 1 Impaired patient. If there are 2x more Normal patients, this would be 2.0. This number gets passed to XGBoost so it pays extra attention to Impaired patients during training.

```python
param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}
```
The "menu" of settings to try. Each one controls something different about the model:
- `n_estimators` - how many trees to build (more = potentially more accurate but slower)
- `max_depth` - how many decisions each tree can make (deeper = more complex patterns but risks overfitting)
- `learning_rate` - how much each tree corrects the previous one (smaller = more cautious, usually better but needs more trees)
- `subsample` - what fraction of rows each tree sees (less than 1.0 adds randomness, reduces overfitting)
- `colsample_bytree` - what fraction of columns each tree sees (same idea as subsample but for columns)
- `min_child_weight` - minimum number of patients a leaf node must cover (higher = simpler model)
- `gamma` - minimum improvement required to make a split (higher = more conservative)

```python
xgb_base = XGBClassifier(
    scale_pos_weight=scale_ratio,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
```
Creates the base XGBoost model with some fixed settings:
- `scale_pos_weight` - the class imbalance ratio from earlier
- `eval_metric='logloss'` - how the model internally measures its own performance during training
- `use_label_encoder=False` - just tells XGBoost not to auto-encode labels (avoids a warning)

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```
Sets up 5-fold cross-validation. This splits the TRAINING data into 5 equal parts. The model trains on 4 parts and tests on the 5th, then rotates, 5 times total. This gives a more reliable accuracy estimate than a single split. `Stratified` keeps the Normal/Impaired ratio the same in each fold.

```python
search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=1
)
```
This is the tuning engine:
- Takes the base model and the param_grid
- `n_iter=30` - randomly picks 30 combinations from the grid (out of 3x4x3x3x3x3x3 = 2,187 possible ones)
- Tests each one using the 5-fold CV from above
- `scoring='accuracy'` - picks the winner based on accuracy
- `n_jobs=-1` - uses all CPU cores to speed things up
- `verbose=1` - prints progress so you know it's working

```python
search.fit(X_train, y_train)
```
Actually runs the search. This is the line that takes a few minutes. It trains 30 x 5 = 150 models total.

```python
best_model = search.best_estimator_
```
Grabs the model that scored the highest accuracy during the search. This is the one that gets used for everything after this point.

---
## Cell 6: Results on test set

```python
y_pred = best_model.predict(X_test)
```
Feeds the 20% test set (that the model has never seen) into the best model. For each patient, it outputs either 0 (Normal) or 1 (Impaired).

```python
print("Test Accuracy:", accuracy_score(y_test, y_pred))
```
Compares the model's guesses (`y_pred`) against the real answers (`y_test`) and prints the percentage it got right.

```python
print(classification_report(y_test, y_pred, target_names=['Normal', 'Impaired']))
```
Prints a more detailed breakdown:
- **Precision** - of everyone the model called Impaired, what % actually were?
- **Recall** - of everyone who actually was Impaired, what % did the model catch?
- **F1-score** - the balance between precision and recall

```python
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Impaired'])
disp.plot(cmap='Blues')
```
Creates and draws the confusion matrix (the 2x2 grid). `cmap='Blues'` just makes it blue-colored.

---
## Cell 7: SHAP explanation

```python
explainer = shap.TreeExplainer(best_model)
```
Creates a SHAP explainer specifically designed for tree-based models (like XGBoost). It knows how to look inside the trees and calculate each feature's contribution.

```python
X_test_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
```
Grabs a random sample of 1,000 patients from the test set. SHAP calculations are slow (it has to analyze every feature for every patient), so using all patients would take too long. 1,000 is enough to see clear patterns.

```python
shap_values = explainer.shap_values(X_test_sample)
```
The actual calculation. For each of the 1,000 patients and each of the 37 features, it computes a number: how much did this feature push the prediction toward Normal (negative) or Impaired (positive)? The result is a big table of 1,000 rows x 37 columns.

```python
shap.summary_plot(shap_values, X_test_sample)
```
Draws the colored dot chart. Each row is a feature, each dot is a patient, color is the feature's value (red=high, blue=low), horizontal position is the impact on the prediction.

---
## Cell 8: Bar chart

```python
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', max_display=15)
```
Same SHAP values, but displayed as a simple bar chart instead of the dot plot. `plot_type='bar'` switches the style. `max_display=15` only shows the top 15 features. Each bar represents the average absolute SHAP value for that feature across all 1,000 patients. Longer bar = more important feature.